# Python Fundamentals: Interactive Lab

এই notebook-এ Day 1-এর Python execution ও object model-এর গুরুত্বপূর্ণ ধারণাগুলো নিজে code চালিয়ে পরীক্ষা করবে।

**কীভাবে ব্যবহার করবে**
1. Google Colab-এ notebook খুলে প্রতিটি cell-এর ▶️ Run চাপো।
2. Output দেখার আগে কী হবে অনুমান করো।
3. Exercise cell-এ TODO পূরণ করো।
4. Python 3 runtime ব্যবহার করো।

এই notebook-এর সব example Python standard library দিয়ে চলে।

## 1. Python implementation এবং runtime

Python language ও Python implementation এক জিনিস নয়। এই cell-এ বর্তমান runtime কোন implementation এবং কোন version তা দেখো।

In [ ]:
import platform
import sys

print("Implementation:", platform.python_implementation())
print("Python version:", sys.version)
print("Executable:", sys.executable)

## 2. Source code → AST → Bytecode

AST হলো source code-এর structural representation। `compile()` code object তৈরি করে। `dis` bytecode inspect করতে সাহায্য করে।

মনে রাখবে: CPython bytecode CPU machine code নয়।

In [ ]:
import ast
import dis

source = """
price = 100
quantity = 3
total = price * quantity
"""

tree = ast.parse(source)
print("AST:")
print(ast.dump(tree, indent=2))

code = compile(source, filename="<lab>", mode="exec")
print("\\nBytecode:")
dis.dis(code)

namespace = {}
exec(code, namespace)
print("\\nExecuted result:", namespace["total"])

## 3. Name binding এবং object identity

Prediction: `second = first` করার পরে list-এ mutation করলে কোন variable-এ পরিবর্তন দেখা যাবে?

In [ ]:
first = ["red", "blue"]
second = first
third = ["red", "blue"]

print("first == third:", first == third)
print("first is third:", first is third)
print("first is second:", first is second)

second.append("green")
print("first:", first)
print("third:", third)

## 4. Mutation বনাম rebinding

Mutable object mutate করা আর variable-কে অন্য object-এ rebind করা এক কাজ নয়।

In [ ]:
items = [1, 2]
before_id = id(items)
items.append(3)
print("After mutation:", items)
print("Same identity:", id(items) == before_id)

count = 10
old_count = count
count = 11
print("Old integer value:", old_count)
print("New integer value:", count)

## 5. Tuple এবং nested mutable object

Tuple-এর item reference বদলানো যাবে না। কিন্তু tuple-এর ভেতরে থাকা list আলাদা mutable object।

In [ ]:
record = ([10, 20], "python")

record[0].append(30)
print(record)

# নিচের লাইন uncomment করলে TypeError হবে:
# record[0] = [100, 200]

## 6. Function argument: mutation বনাম local rebinding

In [ ]:
def mutate(values):
    values.append("added")

def rebind(values):
    values = ["new", "list"]

data = ["original"]
mutate(data)
print("After mutate:", data)

rebind(data)
print("After rebind:", data)

## 7. Mutable default argument bug

Prediction: দ্বিতীয় call-এর output কী হবে? কেন?

In [ ]:
def collect(item, bucket=[]):
    bucket.append(item)
    return bucket

print(collect("A"))
print(collect("B"))
print("Defaults:", collect.__defaults__)

### Safe pattern

In [ ]:
def collect_safely(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(collect_safely("A"))
print(collect_safely("B"))

## 8. Shallow copy বনাম deep copy

In [ ]:
import copy

original = {"scores": [70, 80]}
shallow = copy.copy(original)
deep = copy.deepcopy(original)

print("Outer object shared?", original is shallow)
print("Nested list shared in shallow copy?", original["scores"] is shallow["scores"])
print("Nested list shared in deep copy?", original["scores"] is deep["scores"])

shallow["scores"].append(90)
print("Original after shallow mutation:", original)

deep["scores"].append(100)
print("Original after deep mutation:", original)
print("Deep copy:", deep)

## 9. Dictionary lookup, hash collision এবং equality

Dictionary শুধু hash-এর সঙ্গে value map করে না। Entry-তে প্রকৃত key ও value থাকে। Hash candidate location খুঁজতে সাহায্য করে; collision হলে equality দিয়ে key যাচাই করা হয়। নিচের class ইচ্ছা করে একই hash ফেরত দেয়।

In [ ]:
class CollisionKey:
    def __init__(self, label):
        self.label = label

    def __hash__(self):
        return 7  # ইচ্ছাকৃত hash collision

    def __eq__(self, other):
        return isinstance(other, CollisionKey) and self.label == other.label

first = CollisionKey("first")
second = CollisionKey("second")
mapping = {first: "First value", second: "Second value"}

print(hash(first) == hash(second))  # True
print(first == second)              # False
print(mapping[first])               # First value
print(mapping[second])              # Second value

## 10. Object lifetime এবং hashability

Object lifetime হলো object তৈরি হওয়া থেকে আর অস্তিত্বে না থাকা পর্যন্ত সময়। একটি name delete করলেই object শেষ হয় না, যদি অন্য reference থাকে। Hashable object-এর hash এই lifetime জুড়ে স্থিতিশীল থাকতে হবে।

In [ ]:
items = [10, 20]
alias = items
original_id = id(items)

del items  # name সরানো হলো, object নয়
print(alias)
print(id(alias) == original_id)  # True

### Mutable custom object: কখন hashable হতে পারে?

Mutable object মানেই unhashable নয়। Default custom class equality/hash identity-based হতে পারে। কিন্তু mutable field-এর ওপর __eq__ define করলে hash contract নিয়ে সতর্ক হতে হয়।

In [ ]:
class IdentityUser:
    def __init__(self, name):
        self.name = name

user = IdentityUser("Shomi")
user_set = {user}
user.name = "Shomi Khan"
print(user in user_set)  # True

class ValueUser:
    def __init__(self, user_id):
        self.user_id = user_id

    def __eq__(self, other):
        return isinstance(other, ValueUser) and self.user_id == other.user_id

value_user = ValueUser(10)
try:
    hash(value_user)
except TypeError as error:
    print(type(error).__name__, error)

## 9. Practice: নিজে সমাধান করো

### Exercise A
একটি list-এর alias তৈরি করো। Alias দিয়ে item append করো। Original variable-এ পরিবর্তন দেখা যায় কি?

### Exercise B
একটি string variable-এ নতুন string assign করো। আগের string object-এর value কি বদলায়?

### Exercise C
একটি function লেখো যা dictionary argument-এ `"active": True` যোগ করে। Caller-এর dictionary পরিবর্তিত হয় কি?

### Exercise D
একটি nested dictionary shallow copy করো। Nested list-এ mutation করে behavior ব্যাখ্যা করো।

### Exercise E
একটি mutable default argument bug তৈরি করো এবং `None` sentinel ব্যবহার করে fix করো।

### Exercise F
নিজের একটি function-এর bytecode `dis.dis()` দিয়ে inspect করো।

## Self-check
- [ ] Python implementation ও version identify করতে পারি।
- [ ] AST এবং bytecode-এর পার্থক্য বুঝি।
- [ ] Assignment ও copying এক নয় বুঝি।
- [ ] Mutation ও rebinding আলাদা করতে পারি।
- [ ] Mutable default argument bug ব্যাখ্যা ও fix করতে পারি।
- [ ] Shallow ও deep copy-এর trade-off বুঝি।